## Set-Up

### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data
from helpers import evaluate_forecast

In [2]:
df = load_clean_data()
df = df.sort_values('Date')
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


### Train Test Split

In [3]:
train = df[df['Date'] < '2015-06-30']
test  = df[df['Date'] >= '2015-06-30']

length_check = len(train) + len(test) == len(df)

print(f"Length check: {length_check}")

Length check: True


## Single Store Baselines (Store 1)

### Extract Store 1 Data

In [4]:
store_1_train = train[train['Store'] == 1]
store_1_test  = test[test['Store'] == 1]

print(f"Store 1 train shape: {store_1_train.shape}")
print(f"Store 1 test shape: {store_1_test.shape}")

Store 1 train shape: (910, 18)
Store 1 test shape: (32, 18)


### Naive

In [5]:
store_1_naive_train = store_1_train[store_1_train['Date'] >= '2015-06-23']

print(f"Store 1 naive train shape (last 7 days): {store_1_naive_train.shape}")
print(f"Store 1 naive train shape (last 7 days): {store_1_naive_train['Date'].min()} to {store_1_naive_train['Date'].max()}")

Store 1 naive train shape (last 7 days): (7, 18)
Store 1 naive train shape (last 7 days): 2015-06-23 00:00:00 to 2015-06-29 00:00:00


In [6]:
store_1_naive_forecast = store_1_naive_train[['Store', 'DayOfWeek', 'Sales']].copy()
print(store_1_naive_forecast)


       Store  DayOfWeek  Sales
42370      1          2   3762
41255      1          3   3346
40140      1          4   3533
39025      1          5   3317
37910      1          6   4019
36795      1          7      0
35680      1          1   5197


In [7]:
store_1_naive_forecast = store_1_naive_forecast.rename(columns={'Sales': 'NaiveForecast'})

store_1_test = store_1_test.merge(store_1_naive_forecast, on=['Store', 'DayOfWeek'], how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['DayOfWeek', 'Date', 'Sales', 'NaiveForecast']].head(10))

(32, 19)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
NaiveForecast                0
dtype: int64

   DayOfWeek       Date  Sales  NaiveForecast
0          2 2015-06-30   5735           3762
1          3 2015-07-01   5223           3346
2          4 2015-07-02   5558           3533
3          5 2015-07-03   4665           3317
4          6 2015-07-04   4797           4019
5          7 2015-07-05      0              0
6          1 2015-07-06   4359           5197
7          2 2015-0

#### eval

In [8]:
evaluate_forecast(store_1_test['Sales'], store_1_test['NaiveForecast'])

RMSE: 1029.52
MAE: 797.59
WAPE: 20.10%


### Seasonal Average

#### Last 28-Day Average

In [9]:
store_1_28_day_train = store_1_train[store_1_train['Date'] >= '2015-06-02']

print(f"Store 1 train shape (last 28 days): {store_1_28_day_train.shape}")
print(f"Store 1 train shape (last 28 days): {store_1_28_day_train['Date'].min()} to {store_1_28_day_train['Date'].max()}")

Store 1 train shape (last 28 days): (28, 18)
Store 1 train shape (last 28 days): 2015-06-02 00:00:00 to 2015-06-29 00:00:00


In [10]:
store_1_28_day_train.groupby(['Store', 'DayOfWeek']).size()

Store  DayOfWeek
1      1            4
       2            4
       3            4
       4            4
       5            4
       6            4
       7            4
dtype: int64

In [ ]:
store_1_seasonal_average_28_day_forecast = store_1_28_day_train.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(store_1_seasonal_average_28_day_forecast)

store_1_seasonal_average_28_day_forecast = store_1_seasonal_average_28_day_forecast.reset_index()

Store  DayOfWeek
1      1            4658.00
       2            4541.50
       3            4186.50
       4            2951.25
       5            4149.50
       6            4138.75
       7               0.00
Name: Sales, dtype: float64


In [ ]:
store_1_seasonal_average_28_day_forecast = store_1_seasonal_average_28_day_forecast.rename(columns={'Sales': 'SeasonalAverage28Day'})

store_1_test = store_1_test.merge(store_1_seasonal_average_28_day_forecast, on=['Store', 'DayOfWeek'], how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['DayOfWeek', 'Date', 'Sales', 'SeasonalAverage28Day']].head(10))

(32, 20)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
NaiveForecast                0
28DaySeasonalAverage         0
dtype: int64

   DayOfWeek       Date  Sales  28DaySeasonalAverage
0          2 2015-06-30   5735               4541.50
1          3 2015-07-01   5223               4186.50
2          4 2015-07-02   5558               2951.25
3          5 2015-07-03   4665               4149.50
4          6 2015-07-04   4797               4138.75
5          7 2015-07-05      0        

##### eval

In [ ]:
evaluate_forecast(store_1_test['Sales'], store_1_test['SeasonalAverage28Day'])

RMSE: 910.90
MAE: 704.87
WAPE: 17.76%


#### Full History Average

In [ ]:
store_1_seasonal_average_full_forecast = store_1_train.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(store_1_seasonal_average_full_forecast)

store_1_seasonal_average_full_forecast = store_1_seasonal_average_full_forecast.reset_index()

Store  DayOfWeek
1      1            4945.153846
       2            4652.907692
       3            4456.315385
       4            4077.700000
       5            4518.630769
       6            4963.546154
       7               0.000000
Name: Sales, dtype: float64


In [ ]:
store_1_seasonal_average_full_forecast = store_1_seasonal_average_full_forecast.rename(columns={'Sales': 'FullSeasonalAverage'})

store_1_test = store_1_test.merge(store_1_seasonal_average_full_forecast, on=['Store', 'DayOfWeek'], how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['DayOfWeek', 'Date', 'Sales', 'FullSeasonalAverage']].head(10))

(32, 21)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
NaiveForecast                0
28DaySeasonalAverage         0
FullSeasonalAverage          0
dtype: int64

   DayOfWeek       Date  Sales  FullSeasonalAverage
0          2 2015-06-30   5735          4652.907692
1          3 2015-07-01   5223          4456.315385
2          4 2015-07-02   5558          4077.700000
3          5 2015-07-03   4665          4518.630769
4          6 2015-07-04   4797          4963.546154
5          7 

##### eval

In [16]:
evaluate_forecast(store_1_test['Sales'], store_1_test['FullSeasonalAverage'])

RMSE: 702.58
MAE: 567.18
WAPE: 14.29%


### Single Store Baselines Scores:

| Model | RMSE | MAE | WAPE |
| --- | --- | --- | --- |
| Naive | 1029.52 | 797.59 | 20.10% |
| 28DaySeasonalAverage | 910.90 | 704.87 | 17.76% |
| FullSeasonalAverage | 702.58 | 567.18 | 14.29% |

## Multi-Store Baselines

### Naive

In [17]:
naive_train = train[train['Date'] >= '2015-06-23']

print(f"Naive train shape (last 7 days): {naive_train.shape}")
print(f"Naive train shape (last 7 days): {naive_train['Date'].min()} to {naive_train['Date'].max()}")

Naive train shape (last 7 days): (7805, 18)
Naive train shape (last 7 days): 2015-06-23 00:00:00 to 2015-06-29 00:00:00


In [18]:
naive_forecast = naive_train[['Store', 'DayOfWeek', 'Sales']].copy()
print(naive_forecast)

       Store  DayOfWeek  Sales
42738    369          2   4549
42739    370          2   6021
42740    371          2   4444
42741    372          2   7899
42742    373          2   3609
...      ...        ...    ...
36423    744          1  11682
36424    745          1  14881
36425    746          1   9385
36443    764          1   5941
36420    741          1  12054

[7805 rows x 3 columns]


In [19]:
naive_forecast = naive_forecast.rename(columns={'Sales': 'NaiveForecast'})

print(test.shape)

test = test.merge(naive_forecast, on=['Store', 'DayOfWeek'], how='left')

print(test.shape)
print()
print(test.isnull().sum())
print()
print(test[['Store', 'DayOfWeek', 'Date', 'Sales', 'NaiveForecast']].head(10))

(35680, 18)
(35680, 19)

Store                            0
DayOfWeek                        0
Date                             0
Sales                            0
Customers                        0
Open                             0
Promo                            0
StateHoliday                     0
SchoolHoliday                    0
StoreType                        0
Assortment                       0
CompetitionDistance              0
CompetitionOpenSinceMonth    11328
CompetitionOpenSinceYear     11328
Promo2                           0
Promo2SinceWeek                  0
Promo2SinceYear                  0
PromoInterval                    0
NaiveForecast                    0
dtype: int64

   Store  DayOfWeek       Date  Sales  NaiveForecast
0    369          2 2015-06-30   8172           4549
1    370          2 2015-06-30  14142           6021
2    371          2 2015-06-30   9621           4444
3    372          2 2015-06-30  15550           7899
4    373          2 2015-06-30 

#### eval

In [22]:
store1_check = test[test['Store'] == 1]

print(len(store1_check))

evaluate_forecast(store1_check['Sales'], store1_check['NaiveForecast'])

32
RMSE: 1029.52
MAE: 797.59
WAPE: 20.10%


In [20]:
evaluate_forecast(test['Sales'], test['NaiveForecast'])

RMSE: 2273.06
MAE: 1509.24
WAPE: 24.06%


### Seasonal Average

#### 28-Day Seasonal Average

In [23]:
train_28_day = train[train['Date'] >= '2015-06-02']

print(f"Train shape (last 28 days): {train_28_day.shape}")
print(f"Train shape (last 28 days): {train_28_day['Date'].min()} to {train_28_day['Date'].max()}")

Train shape (last 28 days): (31220, 18)
Train shape (last 28 days): 2015-06-02 00:00:00 to 2015-06-29 00:00:00


In [36]:
train_28_day.groupby(['Store', 'DayOfWeek']).size().value_counts()

4    7805
Name: count, dtype: int64

In [32]:
seasonal_average_28_day_forecast = train_28_day.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(seasonal_average_28_day_forecast)

seasonal_average_28_day_forecast = seasonal_average_28_day_forecast.reset_index()

Store  DayOfWeek
1      1            4658.00
       2            4541.50
       3            4186.50
       4            2951.25
       5            4149.50
                     ...   
1115   3            7033.25
       4            4431.50
       5            7647.50
       6            7208.00
       7               0.00
Name: Sales, Length: 7805, dtype: float64


In [ ]:
seasonal_average_28_day_forecast = seasonal_average_28_day_forecast.rename(columns={'Sales': 'SeasonalAverage28Day'})

test = test.merge(seasonal_average_28_day_forecast, on=['Store', 'DayOfWeek'], how='left')

print(test.shape)
print()
print(test.isnull().sum())
print()
print(test[['Store', 'DayOfWeek', 'Date', 'Sales', 'SeasonalAverage28Day']].head(10))

(35680, 20)

Store                            0
DayOfWeek                        0
Date                             0
Sales                            0
Customers                        0
Open                             0
Promo                            0
StateHoliday                     0
SchoolHoliday                    0
StoreType                        0
Assortment                       0
CompetitionDistance              0
CompetitionOpenSinceMonth    11328
CompetitionOpenSinceYear     11328
Promo2                           0
Promo2SinceWeek                  0
Promo2SinceYear                  0
PromoInterval                    0
NaiveForecast                    0
28DaySeasonalAverage             0
dtype: int64

   Store  DayOfWeek       Date  Sales  28DaySeasonalAverage
0    369          2 2015-06-30   8172               6147.00
1    370          2 2015-06-30  14142               7756.00
2    371          2 2015-06-30   9621               6874.25
3    372          2 2015-06-30  1

##### eval

In [ ]:
store1_check = test[test['Store'] == 1]

print(len(store1_check))

evaluate_forecast(store1_check['Sales'], store1_check['SeasonalAverage28Day'])

32
RMSE: 910.90
MAE: 704.87
WAPE: 17.76%


In [ ]:
evaluate_forecast(test['Sales'], test['SeasonalAverage28Day'])

RMSE: 1646.07
MAE: 1161.31
WAPE: 18.51%


#### Full History Seasonal Average

In [38]:
seasonal_average_full_forecast = train.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(seasonal_average_full_forecast)

seasonal_average_full_forecast = seasonal_average_full_forecast.reset_index()

Store  DayOfWeek
1      1            4945.153846
       2            4652.907692
       3            4456.315385
       4            4077.700000
       5            4518.630769
                       ...     
1115   3            5632.984615
       4            5345.023077
       5            6173.638462
       6            6767.430769
       7               0.000000
Name: Sales, Length: 7805, dtype: float64


In [39]:
seasonal_average_full_forecast = seasonal_average_full_forecast.rename(columns={'Sales': 'FullSeasonalAverage'})

test = test.merge(seasonal_average_full_forecast, on=['Store', 'DayOfWeek'], how='left')

print(test.shape)
print()
print(test.isnull().sum())
print()
print(test[['Store', 'DayOfWeek', 'Date', 'Sales', 'FullSeasonalAverage']].head(10))

(35680, 21)

Store                            0
DayOfWeek                        0
Date                             0
Sales                            0
Customers                        0
Open                             0
Promo                            0
StateHoliday                     0
SchoolHoliday                    0
StoreType                        0
Assortment                       0
CompetitionDistance              0
CompetitionOpenSinceMonth    11328
CompetitionOpenSinceYear     11328
Promo2                           0
Promo2SinceWeek                  0
Promo2SinceYear                  0
PromoInterval                    0
NaiveForecast                    0
28DaySeasonalAverage             0
FullSeasonalAverage              0
dtype: int64

   Store  DayOfWeek       Date  Sales  FullSeasonalAverage
0    369          2 2015-06-30   8172          6275.176923
1    370          2 2015-06-30  14142          7078.907692
2    371          2 2015-06-30   9621          6672.561538
3 

##### eval

In [40]:
store_1_check = test[test['Store'] == 1]

print(len(store_1_check))

evaluate_forecast(store_1_check['Sales'], store_1_check['FullSeasonalAverage'])

32
RMSE: 702.58
MAE: 567.18
WAPE: 14.29%


In [41]:
evaluate_forecast(test['Sales'], test['FullSeasonalAverage'])

RMSE: 1615.74
MAE: 1129.13
WAPE: 18.00%


### All Stores Baselines Scores:

| Model | RMSE | MAE | WAPE |
| --- | --- | --- | --- |
| Naive | 2273.06 | 1509.24 | 24.06% |
| 28DaySeasonalAverage | 1646.07 | 1161.31 | 18.51% |
| FullSeasonalAverage | 1615.74 | 1129.13 | 18.00% |